# 6DoF Object Pose Estimation with Pose Transformer

This notebook demonstrates training the Pose Transformer for **object pose estimation**.
Given an RGB image of an object, the model predicts its 6DoF pose:
- **Rotation**: 3D orientation in SO(3)
- **Translation**: 3D position in R(3)

## Applications
- Robotic manipulation and grasping
- Augmented reality
- Autonomous driving (detecting vehicle poses)
- Industrial inspection

## Datasets Used (Synthetic Examples)
We'll create synthetic pose data for demonstration. In practice, you would use:
- **LINEMOD**: Textured 3D objects with ground truth poses
- **YCB-Video**: Household objects in cluttered scenes
- **T-LESS**: Texture-less industrial objects
- **BOP Challenge**: Benchmark for 6DoF pose estimation

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.transform import Rotation
from tqdm.auto import tqdm
import random

# Set seeds
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Synthetic Object Pose Dataset

We create a synthetic dataset where:
- Images are rendered views of simple 3D shapes
- Ground truth poses are known exactly
- This allows controlled evaluation of the model

In practice, you would load real datasets like LINEMOD or YCB-Video.

In [ ]:
class SyntheticObjectPoseDataset(Dataset):
    """
    Synthetic dataset for object pose estimation.
    
    Creates images with projected 3D wireframe shapes and known poses.
    This simulates the task of estimating pose from visual appearance.
    """
    
    def __init__(self, num_samples=10000, image_size=224, noise_level=0.1):
        self.num_samples = num_samples
        self.image_size = image_size
        self.noise_level = noise_level
        
        # Pre-generate poses for reproducibility
        self.rotations = []
        self.translations = []
        
        for _ in range(num_samples):
            # Random rotation (uniformly distributed on SO(3))
            rot = Rotation.random()
            self.rotations.append(rot.as_matrix())
            
            # Random translation (object center in normalized coords)
            # x, y in [-0.3, 0.3], z in [0.5, 2.0] (depth)
            trans = np.array([
                np.random.uniform(-0.3, 0.3),
                np.random.uniform(-0.3, 0.3),
                np.random.uniform(0.5, 2.0)
            ])
            self.translations.append(trans)
        
        self.rotations = np.array(self.rotations, dtype=np.float32)
        self.translations = np.array(self.translations, dtype=np.float32)
        
        # Define a simple 3D cube for visualization
        self.cube_vertices = np.array([
            [-1, -1, -1], [1, -1, -1], [1, 1, -1], [-1, 1, -1],
            [-1, -1, 1], [1, -1, 1], [1, 1, 1], [-1, 1, 1]
        ], dtype=np.float32) * 0.1  # Scale down
        
        self.cube_edges = [
            (0, 1), (1, 2), (2, 3), (3, 0),  # Back face
            (4, 5), (5, 6), (6, 7), (7, 4),  # Front face
            (0, 4), (1, 5), (2, 6), (3, 7)   # Connecting edges
        ]
    
    def __len__(self):
        return self.num_samples
    
    def _project_points(self, points_3d, rotation, translation, K=None):
        """Project 3D points to 2D using perspective projection."""
        if K is None:
            # Simple camera intrinsics
            fx = fy = self.image_size
            cx = cy = self.image_size / 2
            K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])
        
        # Transform points
        points_cam = (rotation @ points_3d.T).T + translation
        
        # Project
        points_2d = (K @ points_cam.T).T
        points_2d = points_2d[:, :2] / points_2d[:, 2:3]
        
        return points_2d
    
    def _render_wireframe(self, rotation, translation):
        """Render a wireframe cube with given pose."""
        image = np.zeros((self.image_size, self.image_size, 3), dtype=np.float32)
        
        # Project vertices
        points_2d = self._project_points(self.cube_vertices, rotation, translation)
        
        # Draw edges
        for i, (start, end) in enumerate(self.cube_edges):
            p1 = points_2d[start].astype(int)
            p2 = points_2d[end].astype(int)
            
            # Simple line drawing
            self._draw_line(image, p1, p2, color=(1.0, 0.5, 0.0))
        
        # Draw vertices
        for i, p in enumerate(points_2d):
            x, y = int(p[0]), int(p[1])
            if 0 <= x < self.image_size and 0 <= y < self.image_size:
                # Color front vertices differently
                color = (0.0, 1.0, 0.0) if i >= 4 else (1.0, 0.0, 0.0)
                self._draw_circle(image, x, y, radius=3, color=color)
        
        # Add noise
        if self.noise_level > 0:
            noise = np.random.randn(*image.shape).astype(np.float32) * self.noise_level
            image = np.clip(image + noise, 0, 1)
        
        return image
    
    def _draw_line(self, image, p1, p2, color):
        """Draw a line using Bresenham's algorithm."""
        x1, y1 = p1
        x2, y2 = p2
        
        dx = abs(x2 - x1)
        dy = abs(y2 - y1)
        sx = 1 if x1 < x2 else -1
        sy = 1 if y1 < y2 else -1
        err = dx - dy
        
        while True:
            if 0 <= x1 < self.image_size and 0 <= y1 < self.image_size:
                image[y1, x1] = color
            
            if x1 == x2 and y1 == y2:
                break
            
            e2 = 2 * err
            if e2 > -dy:
                err -= dy
                x1 += sx
            if e2 < dx:
                err += dx
                y1 += sy
    
    def _draw_circle(self, image, cx, cy, radius, color):
        """Draw a filled circle."""
        for y in range(max(0, cy - radius), min(self.image_size, cy + radius + 1)):
            for x in range(max(0, cx - radius), min(self.image_size, cx + radius + 1)):
                if (x - cx) ** 2 + (y - cy) ** 2 <= radius ** 2:
                    image[y, x] = color
    
    def __getitem__(self, idx):
        rotation = self.rotations[idx]
        translation = self.translations[idx]
        
        # Render image
        image = self._render_wireframe(rotation, translation)
        
        # Convert to tensor (C, H, W)
        image = torch.from_numpy(image).permute(2, 0, 1)
        rotation = torch.from_numpy(rotation)
        translation = torch.from_numpy(translation)
        
        return image, rotation, translation


# Create datasets
train_dataset = SyntheticObjectPoseDataset(num_samples=5000, noise_level=0.05)
val_dataset = SyntheticObjectPoseDataset(num_samples=1000, noise_level=0.05)

print(f'Training samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')

In [ ]:
# Visualize samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(8):
    image, rotation, translation = train_dataset[i]
    ax = axes[i // 4, i % 4]
    ax.imshow(image.permute(1, 2, 0).numpy())
    ax.set_title(f'z={translation[2]:.2f}')
    ax.axis('off')

plt.suptitle('Synthetic Object Pose Dataset Samples', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Pose Transformer Model

The Pose Transformer takes images as input and predicts:
- **Rotation**: Using 6D continuous representation (Zhou et al., CVPR 2019)
- **Translation**: 3D vector in camera coordinates

In [ ]:
from examples.pose_transformer import (
    PoseTransformer,
    pose_loss,
    geodesic_loss_rotmat,
    rotation_6d_to_matrix,
)

# Create model
model = PoseTransformer(
    image_size=224,
    patch_size=16,
    in_channels=3,
    embed_dim=192,  # Tiny variant
    depth=6,
    num_heads=3,
    rotation_repr='6d',
    num_objects=1,
    translation_scale=1.0,
)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

## 3. Training Setup

In [ ]:
# Hyperparameters
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS = 20
WEIGHT_DECAY = 1e-4
ROTATION_WEIGHT = 1.0
TRANSLATION_WEIGHT = 10.0  # Scale since translation values are smaller

# Data loaders
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [ ]:
def train_epoch(model, loader, optimizer, device, rot_weight=1.0, trans_weight=1.0):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    total_rot_loss = 0
    total_trans_loss = 0
    num_samples = 0
    
    pbar = tqdm(loader, desc='Training')
    for images, gt_rotation, gt_translation in pbar:
        images = images.to(device)
        gt_rotation = gt_rotation.to(device)
        gt_translation = gt_translation.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        pred_rotation, pred_translation = model(images)
        
        # Compute loss
        loss, rot_loss, trans_loss = pose_loss(
            pred_rotation, pred_translation,
            gt_rotation, gt_translation,
            rotation_repr='6d',
            rotation_weight=rot_weight,
            translation_weight=trans_weight,
        )
        
        loss.backward()
        optimizer.step()
        
        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_rot_loss += rot_loss.item() * batch_size
        total_trans_loss += trans_loss.item() * batch_size
        num_samples += batch_size
        
        pbar.set_postfix({
            'loss': loss.item(),
            'rot': rot_loss.item(),
            'trans': trans_loss.item()
        })
    
    return {
        'loss': total_loss / num_samples,
        'rot_loss': total_rot_loss / num_samples,
        'trans_loss': total_trans_loss / num_samples,
    }


def evaluate(model, loader, device, rot_weight=1.0, trans_weight=1.0):
    """Evaluate the model."""
    model.eval()
    total_loss = 0
    total_rot_loss = 0
    total_trans_loss = 0
    total_rot_error = 0  # Angular error in degrees
    total_trans_error = 0  # Euclidean distance
    num_samples = 0
    
    with torch.no_grad():
        for images, gt_rotation, gt_translation in tqdm(loader, desc='Evaluating'):
            images = images.to(device)
            gt_rotation = gt_rotation.to(device)
            gt_translation = gt_translation.to(device)
            
            # Forward pass
            pred_rotation, pred_translation = model(images)
            
            # Compute loss
            loss, rot_loss, trans_loss = pose_loss(
                pred_rotation, pred_translation,
                gt_rotation, gt_translation,
                rotation_repr='6d',
                rotation_weight=rot_weight,
                translation_weight=trans_weight,
            )
            
            # Convert to matrices for error computation
            pred_mat = rotation_6d_to_matrix(pred_rotation)
            
            # Angular error (geodesic distance)
            diff = torch.bmm(pred_mat.transpose(-2, -1), gt_rotation)
            trace = diff[:, 0, 0] + diff[:, 1, 1] + diff[:, 2, 2]
            cos_angle = (trace - 1.0) / 2.0
            cos_angle = torch.clamp(cos_angle, -1.0, 1.0)
            angle_error = torch.acos(cos_angle) * 180.0 / np.pi  # Degrees
            
            # Translation error
            trans_error = torch.norm(pred_translation - gt_translation, dim=-1)
            
            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            total_rot_loss += rot_loss.item() * batch_size
            total_trans_loss += trans_loss.item() * batch_size
            total_rot_error += angle_error.sum().item()
            total_trans_error += trans_error.sum().item()
            num_samples += batch_size
    
    return {
        'loss': total_loss / num_samples,
        'rot_loss': total_rot_loss / num_samples,
        'trans_loss': total_trans_loss / num_samples,
        'rot_error_deg': total_rot_error / num_samples,
        'trans_error': total_trans_error / num_samples,
    }

## 4. Training Loop

In [ ]:
# Training history
history = {
    'train_loss': [], 'train_rot_loss': [], 'train_trans_loss': [],
    'val_loss': [], 'val_rot_loss': [], 'val_trans_loss': [],
    'val_rot_error': [], 'val_trans_error': []
}

best_rot_error = float('inf')

for epoch in range(NUM_EPOCHS):
    print(f'\nEpoch {epoch + 1}/{NUM_EPOCHS}')
    print('-' * 50)
    
    # Train
    train_metrics = train_epoch(
        model, train_loader, optimizer, device,
        rot_weight=ROTATION_WEIGHT, trans_weight=TRANSLATION_WEIGHT
    )
    
    # Evaluate
    val_metrics = evaluate(
        model, val_loader, device,
        rot_weight=ROTATION_WEIGHT, trans_weight=TRANSLATION_WEIGHT
    )
    
    # Update scheduler
    scheduler.step()
    
    # Save history
    history['train_loss'].append(train_metrics['loss'])
    history['train_rot_loss'].append(train_metrics['rot_loss'])
    history['train_trans_loss'].append(train_metrics['trans_loss'])
    history['val_loss'].append(val_metrics['loss'])
    history['val_rot_loss'].append(val_metrics['rot_loss'])
    history['val_trans_loss'].append(val_metrics['trans_loss'])
    history['val_rot_error'].append(val_metrics['rot_error_deg'])
    history['val_trans_error'].append(val_metrics['trans_error'])
    
    print(f"Train - Loss: {train_metrics['loss']:.4f}, "
          f"Rot: {train_metrics['rot_loss']:.4f}, Trans: {train_metrics['trans_loss']:.4f}")
    print(f"Val   - Loss: {val_metrics['loss']:.4f}, "
          f"Rot: {val_metrics['rot_loss']:.4f}, Trans: {val_metrics['trans_loss']:.4f}")
    print(f"Val Errors - Rotation: {val_metrics['rot_error_deg']:.2f} deg, "
          f"Translation: {val_metrics['trans_error']:.4f}")
    
    # Save best model
    if val_metrics['rot_error_deg'] < best_rot_error:
        best_rot_error = val_metrics['rot_error_deg']
        torch.save(model.state_dict(), 'pose_transformer_best.pth')
        print(f'Saved best model with rotation error: {best_rot_error:.2f} deg')

print(f'\nBest Rotation Error: {best_rot_error:.2f} degrees')

## 5. Visualize Training

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Total loss
axes[0, 0].plot(history['train_loss'], label='Train')
axes[0, 0].plot(history['val_loss'], label='Val')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Total Loss')
axes[0, 0].set_title('Total Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Rotation loss (geodesic)
axes[0, 1].plot(history['train_rot_loss'], label='Train')
axes[0, 1].plot(history['val_rot_loss'], label='Val')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Rotation Loss (rad)')
axes[0, 1].set_title('Rotation Loss (Geodesic)')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Rotation error in degrees
axes[1, 0].plot(history['val_rot_error'])
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Rotation Error (degrees)')
axes[1, 0].set_title('Validation Rotation Error')
axes[1, 0].grid(True)

# Translation error
axes[1, 1].plot(history['val_trans_error'])
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Translation Error')
axes[1, 1].set_title('Validation Translation Error')
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

## 6. Visualize Predictions

In [ ]:
def visualize_predictions(model, dataset, device, num_samples=4):
    """Visualize model predictions vs ground truth."""
    model.eval()
    
    fig = plt.figure(figsize=(16, 4 * num_samples))
    
    for i in range(num_samples):
        image, gt_rotation, gt_translation = dataset[i]
        image_tensor = image.unsqueeze(0).to(device)
        
        with torch.no_grad():
            pred_rotation, pred_translation = model(image_tensor)
            pred_mat = rotation_6d_to_matrix(pred_rotation)[0].cpu().numpy()
        
        gt_mat = gt_rotation.numpy()
        pred_trans = pred_translation[0].cpu().numpy()
        gt_trans = gt_translation.numpy()
        
        # Compute errors
        diff = pred_mat.T @ gt_mat
        trace = np.trace(diff)
        angle_error = np.arccos(np.clip((trace - 1) / 2, -1, 1)) * 180 / np.pi
        trans_error = np.linalg.norm(pred_trans - gt_trans)
        
        # Plot image
        ax1 = fig.add_subplot(num_samples, 3, i * 3 + 1)
        ax1.imshow(image.permute(1, 2, 0).numpy())
        ax1.set_title(f'Input Image')
        ax1.axis('off')
        
        # Plot rotation (as coordinate frames)
        ax2 = fig.add_subplot(num_samples, 3, i * 3 + 2, projection='3d')
        
        # Ground truth frame (solid)
        origin = np.zeros(3)
        for j, (color, label) in enumerate(zip(['r', 'g', 'b'], ['X', 'Y', 'Z'])):
            ax2.quiver(*origin, *gt_mat[:, j], color=color, arrow_length_ratio=0.1, linewidth=2)
        
        # Predicted frame (dashed)
        for j, color in enumerate(['r', 'g', 'b']):
            ax2.quiver(*origin, *pred_mat[:, j], color=color, arrow_length_ratio=0.1, 
                      linewidth=2, linestyle='--', alpha=0.6)
        
        ax2.set_xlim([-1, 1])
        ax2.set_ylim([-1, 1])
        ax2.set_zlim([-1, 1])
        ax2.set_title(f'Rotation (Error: {angle_error:.1f}°)')
        
        # Plot translation
        ax3 = fig.add_subplot(num_samples, 3, i * 3 + 3, projection='3d')
        ax3.scatter(*gt_trans, color='green', s=100, label='GT', marker='o')
        ax3.scatter(*pred_trans, color='red', s=100, label='Pred', marker='x')
        ax3.plot([gt_trans[0], pred_trans[0]], 
                 [gt_trans[1], pred_trans[1]], 
                 [gt_trans[2], pred_trans[2]], 'k--', alpha=0.5)
        ax3.set_xlabel('X')
        ax3.set_ylabel('Y')
        ax3.set_zlabel('Z')
        ax3.set_title(f'Translation (Error: {trans_error:.3f})')
        ax3.legend()
    
    plt.tight_layout()
    plt.show()


# Load best model and visualize
model.load_state_dict(torch.load('pose_transformer_best.pth'))
visualize_predictions(model, val_dataset, device, num_samples=4)

## 7. Evaluation Metrics for Pose Estimation

Standard metrics for 6DoF pose estimation:
- **ADD**: Average Distance of model points under predicted vs ground truth pose
- **ADD-S**: Symmetric version for symmetric objects
- **< 5cm, 5°**: Pose accuracy threshold

In [ ]:
def compute_add_metric(pred_rot, pred_trans, gt_rot, gt_trans, model_points):
    """
    Compute ADD (Average Distance of model points) metric.
    
    Args:
        pred_rot: Predicted rotation matrix (3, 3)
        pred_trans: Predicted translation (3,)
        gt_rot: Ground truth rotation (3, 3)
        gt_trans: Ground truth translation (3,)
        model_points: 3D model points (N, 3)
    
    Returns:
        ADD metric value
    """
    # Transform points by predicted pose
    pred_points = (pred_rot @ model_points.T).T + pred_trans
    
    # Transform points by ground truth pose
    gt_points = (gt_rot @ model_points.T).T + gt_trans
    
    # Compute average distance
    distances = np.linalg.norm(pred_points - gt_points, axis=1)
    return np.mean(distances)


def evaluate_with_add(model, dataset, device, model_points, threshold=0.1):
    """Evaluate using ADD metric with a threshold."""
    model.eval()
    
    add_values = []
    successes = 0
    
    for i in tqdm(range(len(dataset)), desc='Computing ADD'):
        image, gt_rotation, gt_translation = dataset[i]
        image_tensor = image.unsqueeze(0).to(device)
        
        with torch.no_grad():
            pred_rotation, pred_translation = model(image_tensor)
            pred_mat = rotation_6d_to_matrix(pred_rotation)[0].cpu().numpy()
        
        gt_mat = gt_rotation.numpy()
        pred_trans = pred_translation[0].cpu().numpy()
        gt_trans = gt_translation.numpy()
        
        add = compute_add_metric(pred_mat, pred_trans, gt_mat, gt_trans, model_points)
        add_values.append(add)
        
        if add < threshold:
            successes += 1
    
    accuracy = 100.0 * successes / len(dataset)
    mean_add = np.mean(add_values)
    
    return {
        'mean_add': mean_add,
        'accuracy': accuracy,
        'threshold': threshold
    }


# Evaluate with ADD metric
model_points = val_dataset.cube_vertices
add_results = evaluate_with_add(model, val_dataset, device, model_points, threshold=0.05)

print(f"\nADD Evaluation Results:")
print(f"Mean ADD: {add_results['mean_add']:.4f}")
print(f"Accuracy (< {add_results['threshold']}): {add_results['accuracy']:.2f}%")

## Summary

This notebook demonstrated:

1. **Synthetic Object Pose Dataset**: Wireframe cubes with random poses
2. **Pose Transformer**: Vision Transformer for 6DoF pose estimation
3. **6D Rotation Representation**: Continuous representation for learning rotations
4. **Geodesic Loss**: Proper loss function for SO(3) rotation learning
5. **ADD Metric**: Standard evaluation metric for pose estimation

### Key Takeaways:
- **6D representation** is better than quaternions for gradient-based learning
- **Geodesic loss** respects the geometry of rotation space
- **ADD metric** measures actual 3D alignment, not just rotation/translation separately

### Next Steps:
- Train on real datasets (LINEMOD, YCB-Video)
- Add object detection for multi-object scenarios
- Incorporate depth information for better translation estimation
- Use render-and-compare refinement for higher accuracy